# Нейросеть для предсказания калорийности блюд


Задача - обучить нейросеть, которая будет предсказывать калорийность блюд по их фотографии и текстовому описанию ингредиентов.


In [ ]:
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from scripts.utils import get_dataset_url, load_dish_image, parse_ingredients

RANDOM_STATE = 42

## Подготовка данных


Загружаем набор данных.


In [ ]:
DATASET_URL = "https://disk.yandex.ru/d/kz9g5msVqtahiw"

download_url = get_dataset_url(DATASET_URL)

!wget -O nutrition.zip -nc "$download_url"

Распаковываем датасет.


In [ ]:
DATA_DIR = Path.cwd() / "data"

if Path(DATA_DIR).exists():
    print("Датасет уже распакован.")
else:
    with zipfile.ZipFile("nutrition.zip") as zip_archive:
        zip_archive.extractall()
        print("Датасет распакован.")

Теперь посмотрим на состав датасета.


In [ ]:
images = DATA_DIR.rglob("rgb.png")
images_count = sum(1 for item in images if item.is_file())

print(f"Количество фотографий блюд: {images_count}")

In [ ]:
DISH_CSV = "dish.csv"
INGREDIENTS_CSV = "ingredients.csv"

dishes_df = pd.read_csv(DATA_DIR / DISH_CSV, index_col="dish_id")
ingredients_df = pd.read_csv(DATA_DIR / INGREDIENTS_CSV, index_col="id")

print(f"Записей в {DISH_CSV}        : {len(dishes_df)}")
print(f"Записей в {INGREDIENTS_CSV} : {len(ingredients_df)}")

Всего в датасете:

- 3490 фотографий блюд,
- 3262 записи с информацией о блюдах и их состав,
- 555 записей об ингредиентах.

In [ ]:
ingredients_df.head()

In [ ]:
dishes_df.head()

Ингредиенты закодированы идентификаторами, которые нужно заменить на названия продуктов. Убираем префикс; оставшееся число будет номером ингредиента в `ingredients.csv`.


In [ ]:
dishes_df["ingredients"] = dishes_df["ingredients"].apply(
    parse_ingredients, ingredients_map=ingredients_df["ingr"].to_dict()
)
dishes_df.head()

## EDA


In [ ]:
dishes_df.info()

В датасете содержится информация о:

- калорийности блюда в килокалориях - целевая переменная,
- масса порции в граммах,
- список ингредиентов через запятую,
- название выборки (train/test).


In [ ]:
dishes_df.isna().sum()

Пропусков в данных нет.


In [ ]:
pd.DataFrame({
    "Количество": dishes_df["split"].value_counts(normalize=False),
    "Доля, %": dishes_df["split"].value_counts(normalize=True),
}).style.format({"Доля, %": "{:.2%}"})

Данные уже разделены в соотношении 85% к 15%: 2755 записей в train, 507 - в test.


### Ингредиенты

In [ ]:
from collections import Counter

all_ingredients = (
    ingredient
    for ingredients in dishes_df["ingredients"]
    for ingredient in ingredients.split(", ")
)

ingredients_count = (
    pd
    .DataFrame(Counter(all_ingredients).items(), columns=["Ингредиент", "Количество"])
    .sort_values(by="Количество", ascending=False)
    .reset_index(drop=True)
)

ingredients_count

Есть ингредиенты, которые часто встречаются, а есть те, которые используются только в одном блюде.

In [ ]:
sns.barplot(
    data=ingredients_count.head(10),
    x="Количество",
    y="Ингредиент",
    orient="h",
    color="skyblue",
)

plt.title("Топ-10 наиболее используемых ингредиентов")
plt.show()

Чаще всего в блюдах используются приправы: оливковое масло, соль, перец, чеснок.

### Калорийность и масса порции

In [ ]:
dishes_df.describe(percentiles=[0.5, 0.95, 0.99]).T

In [ ]:
total_mass_q99 = dishes_df[
    dishes_df["total_mass"] < dishes_df["total_mass"].quantile(0.99)
]["total_mass"]
total_calories_q99 = dishes_df[
    dishes_df["total_calories"] < dishes_df["total_calories"].quantile(0.99)
]["total_calories"]

_, ax = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

sns.histplot(total_mass_q99, kde=True, ax=ax[0], color="skyblue", edgecolor="white")
ax[0].set_title("Распределение масс порций")
ax[0].set_xlabel("Масса порции, г")

sns.histplot(total_calories_q99, kde=True, color="salmon", edgecolor="white")
ax[1].set_title("Распределение калорийностей блюд")
ax[1].set_xlabel("Калорийность, ккал")

plt.tight_layout()
plt.show()

Средние значения для калорийности и массы порции небольшие (255 и 215 соответственно), но есть и экстремальные значения. Это говорит о том, что у распределения есть длинный хвост справа. Редкие большие значения могут искажать предсказания модели для блюд с большой калорийностью.

In [ ]:
mass_calories_corr = dishes_df[["total_mass", "total_calories"]].corr()

plt.figure(figsize=(5, 3))

sns.heatmap(
    mass_calories_corr,
    cmap="coolwarm",
    annot=True,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5},
)

plt.show()

Корреляция между массой порции и количеством калорий равна 0.76 - это указывает на сильную положительную связь между значениями, т.е. с помощью массы можно хорошо предсказывать калорийность.

### Фотографии блюд

In [ ]:
dish_ids = dishes_df.sample(9, random_state=RANDOM_STATE).index.to_list()

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(9, 9))

axes = axes.flatten()

for ax, dish_id in zip(axes, dish_ids):
    img = load_dish_image(dish_id)

    total_calories, total_mass = dishes_df.loc[
        dish_id, ["total_calories", "total_mass"]
    ]

    title = f"{dish_id} | {total_mass:.2f} г. | {total_calories:.2f} ккал"

    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

Фотографии блюд с различным приближением, положением камеры и освещением - нужно учесть при аугментации.

### Выводы по EDA

В датасете предоставлена информация о 3262 блюдах: изображение, текстовое перечисление ингредиентов, масса порции и калорийность (целевая переменная). Пропусков в данных нет. Разделение на train/test выборки уже сделано: 2755 в train, 507 в test.

Распределение для калорийности скошено вправо, среднее 255 ккал, а наибольшее - 3943 ккал. 99% всех значений не превышают 918 ккал. Поэтому, скорее всего модель предсказания может недооценивать очень калорийные блюда. Также, между калорийностью и размером порции есть сильная положительная связь, значит, размер порции будет полезным признаком для модели.

Среди наиболее часто встречающихся ингредиентов есть высококалорийные, так и низкокалорийные ингредиенты. Например, оливковое масло используют в половине всех представленных блюд, но в данных не указывается его количество, поэтому использование этого признака не даёт модели однозначного сигнала для увеличения предсказанного значения. Низкокалорийные ингредиенты также часто используются в блюдах, поэтому из-за них модель также может "повысить" итоговое значение, хотя они почти не вносят свой вклад.

### Решение задачи предсказания калорийности блюда

В этой задаче у нас имеются различные данные (числовые, текстовые) и изображения. Значит, нужно реализовать мультимодальную модель, которая использует их все:
- для изображений используем предобученную модель `EfficientNetV2-M`, которая может извлечь визуальные признаки из фотографий блюд,
- для кодирования признаков по текстовому составу ингредиентов используем модель `all-MiniLM-L6-v2`,
- добавляем проекционные слои для приведения размерностей к общему значению,
- числовые значения масс порций нормализуем с помощью `StandardScaler`,
- полученные эмбеддинги конкатенируем и подаём в линейный слой, которые будет предсказывать калорийность.

Изображения нужно аугментировать перед обучением:
- `RandomResizedCrop` - меняем размеры и масштаб, т.к. фотографии блюд с разным приближением,
- `HorizontalFlip`/`VerticalFlip`/`RandomRotate90` - учитываем разное положение камеры,
- `ColorJitter` - разное освещение
- `CoarseDropout` - для улучшения качества обучения
- `Normalize` - по тем значениям, которые использовались в модели

Данных не очень много, поэтому нужно учесть вероятность переобучения и вовремя остановиться. Для измерения качества обучения используем MAE. Требование - MAE меньше 50 ккал на тестовой выборке.